# Final analysis — master tables from `runs/`

This is the single place that turns saved runs into the report's tables. It scans `runs/` directly,
so experiments from **`02_main_exps_1.ipynb` and `02_main_exps_2.ipynb` are aggregated together** —
run the batch notebooks in any order, then re-run this file top to bottom.

Rules it follows:
- Source of truth = each run's `config.json` + `metrics.csv`. Never report numbers from memory.
- `LEGACY` excludes the 50k-protocol prototypes (exp_000/001/002): 45k and 50k numbers are not comparable.
- Collapsed / bad runs are kept in the table, not filtered out — they are findings (see exp_102 seed 42).
- Rows are sorted by experiment number, so a second batch numbered exp_2xx just appends.

No GPU is needed here; this cell only reads files.

In [1]:
import sys, pathlib, json
import pandas as pd

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":          # work whether the kernel starts in notebooks/ or repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import RUNS_DIR

print("project root:", ROOT)
print("runs dir:    ", RUNS_DIR)

project root: c:\Users\sigridwicks\Desktop\DL HW1
runs dir:     C:\Users\sigridwicks\Desktop\DL HW1\runs


In [2]:
# Master table: every run under runs/ is picked up automatically, across all batch notebooks.
LEGACY = {"exp_000_baseline", "exp_001_adam", "exp_002_adam_45k"}   # 50k-protocol prototypes

rows = []
for d in sorted(RUNS_DIR.glob("exp_*")):
    if not d.is_dir() or d.name in LEGACY or not (d / "config.json").exists():
        continue
    cfg = json.loads((d / "config.json").read_text())
    m = pd.read_csv(d / "metrics.csv")
    rows.append({"run": d.name, "arch": cfg.get("arch"), "optimizer": cfg.get("optimizer"),
                 "lr": cfg.get("lr"), "aug": cfg.get("aug"), "seed": cfg.get("seed"),
                 "epochs": cfg.get("epochs"), "val_top1": round(m["test_acc"].iloc[-1], 4)})

df = pd.DataFrame(rows)
df["exp"] = df["run"].str.replace(r"_s\d+$", "", regex=True)

def exp_num(s):                      # sort by experiment number, not lexicographically
    return int(s.split("_")[1])

df = df.sort_values(by=["exp", "seed"], key=lambda c: c.map(exp_num) if c.name == "exp" else c)
df = df.reset_index(drop=True)

per_exp = df.groupby("exp")["val_top1"].agg(["mean", "std", "count"]).round(4)
per_exp = per_exp.reindex(sorted(per_exp.index, key=exp_num))
per_exp["mean ± std"] = per_exp.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)

display(df)
display(per_exp)
# For the report: df.to_markdown() / per_exp.to_markdown()   (needs `pip install tabulate`)
# Snapshot: per_exp.to_csv(ROOT / "results_master.csv")

,run,arch,optimizer,lr,aug,seed,epochs,val_top1,exp
0,exp_101_baseline_sgd_s0,baseline,SGD,0.01,none,0,10,0.1166,exp_101_baseline_sgd
1,exp_101_baseline_sgd_s42,baseline,SGD,0.01,none,42,10,0.1126,exp_101_baseline_sgd
2,exp_101_baseline_sgd_s123,baseline,SGD,0.01,none,123,10,0.1108,exp_101_baseline_sgd
3,exp_102_adam_s0,baseline,Adam,0.001,none,0,10,0.2794,exp_102_adam
4,exp_102_adam_s42,baseline,Adam,0.001,none,42,10,0.0076,exp_102_adam
...,...,...,...,...,...,...,...,...,...
66,exp_219_wide3_aug_s42,down+wide3,Adam,1e-3 + warmup/cos,crop+flip,42,60,0.7350,exp_219_wide3_aug
67,exp_219_wide3_aug_s123,down+wide3,Adam,1e-3 + warmup/cos,crop+flip,123,60,0.7384,exp_219_wide3_aug
68,exp_220_wide_cutout_s0,down+wide,Adam,1e-3 + warmup/cos,crop+flip+erase,0,60,0.7254,exp_220_wide_cutout
69,exp_220_wide_cutout_s42,down+wide,Adam,1e-3 + warmup/cos,crop+flip+erase,42,60,0.7336,exp_220_wide_cutout


,mean,std,count,mean ± std
exp,,,,
exp_101_baseline_sgd,0.1133,0.0030,3,0.1133 ± 0.0030
exp_102_adam,0.2107,0.1790,3,0.2107 ± 0.1790
exp_103_bn,0.4717,0.0025,3,0.4717 ± 0.0025
exp_104_lrsched,0.5409,0.0011,3,0.5409 ± 0.0011
exp_105_aug,0.4993,0.0018,3,0.4993 ± 0.0018
exp_106_arch_3x3,0.5425,0.0053,3,0.5425 ± 0.0053
exp_107_arch_down,0.6100,0.0082,3,0.6100 ± 0.0082
exp_108_arch_3x3_noaug,0.5786,0.0021,3,0.5786 ± 0.0021
exp_109_arch_down_noaug,0.5929,0.0077,3,0.5929 ± 0.0077
